# Day 24：FastAPI 后端开发(下)+数据库：SSE流式、CORS、SQLAlchemy持久化与全栈联调（Jupyter Notebook 实战手册）

> **所属阶段**：第 2 阶段 · 第二阶段：大模型核心原理、Prompt工程与全栈 Web API ｜ **第 4 周**：第4周：全栈 Web 聊天应用与 FastAPI 后端工程  
> **GitLab 工单**：`STARMIND-024` ｜ **开发分支**：`feat/day24-fastapi-sse-sqlite`  
> **配套 HTML5 课件**：`courseware/day24/index.html` ｜ **独立版本快照**：`versions/day24/`

---

## 🎙️ 老陈开场旁白

早啊各位！今天是第二阶段的**终极收官日**，也是咱们第一个全栈可交互里程碑——「网页版星辰 ChatGPT 克隆」！前几天大家把大模型原理、Prompt、安全、工具调用、前端聊天页面和 FastAPI REST API 全部一块块打磨好了，今天咱们用三大企业级拼图把它们焊成一体：
1. **SQLite + SQLAlchemy 2.x ORM**：建 `chat_sessions` 和 `chat_messages` 两张关联表，把每一轮问答稳稳落盘持久化；
2. **SSE（Server-Sent Events）流式推送**：用 FastAPI `StreamingResponse(..., media_type='text/event-stream')` 实时吐出 `data: {"delta": ...}\n\n` 打字机流，并在流结束后自动把完整回答写入数据库；
3. **CORS 跨域中间件 + 前端页面托管**：配置 `CORSMiddleware` 并挂载首页 HTML，用 `TestClient` 跑通端到端全栈联调！开干！

### 承上启下
- **昨日基线**：昨天（Day 23）我们用 FastAPI + Pydantic v2 把星辰智脑封装成了标准 REST API 并跑通了 `TestClient` 测试。但距离真正的网页版 ChatGPT 还差最后三块拼图：第一，重启服务后内存里的聊天记录全没了；第二，接口还是同步一次性返回，没有 SSE 流式打字机推送；第三，前后端分离部署时会遇到浏览器 CORS 跨域拦截！
- **明日预告**：明天（Day 25）我们将正式迈入第三阶段——LangChain 与企业级 RAG 开发！我们会把这两周手写的模型调用、Prompt 模板与工具调用全面升级为 LangChain 标准化抽象层，并构建企业级知识库问答系统。


In [ ]:
# 环境自检与路径初始化：确保可以无缝导入当日版本模块并支持异步协程运行
import os
import sys
from pathlib import Path

# 允许在 Jupyter 已有事件循环内安全调用 asyncio.run()
try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    pass

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'versions').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DAY_DIR = PROJECT_ROOT / 'versions' / 'day24'
for _p in (DAY_DIR, DAY_DIR / 'src', PROJECT_ROOT):
    if _p.exists() and str(_p) not in sys.path:
        sys.path.insert(0, str(_p))
os.chdir(DAY_DIR)
__file__ = str(DAY_DIR / 'src' / 'notebook_entry.py')

print(f'✅ Python 版本: {sys.version.split()[0]}')
print(f'✅ 项目根目录: {PROJECT_ROOT}')
print(f'✅ 当日版本目录: {DAY_DIR}')


## 📋 企业需求背景与验收目标（STARMIND-024）

星辰智脑 V1.0 网页版工作台需要正式交付内测：员工在浏览器打开即可进行流式对话，刷新页面或重启后端后历史会话完整保留，同时允许公司内部其他前端域名（如 OA 门户）安全跨域调用 API。

**今日迭代目标：**
- 使用 SQLAlchemy 2.x 现代声明式 ORM（`DeclarativeBase` + `Mapped` + `mapped_column`）设计 `ChatSession` 与 `ChatMessage` 一对多数据表并实现 CRUD 仓库层。
- 在 FastAPI 中配置 `CORSMiddleware`，托管企业聊天前端页面 `/`，并实现 `POST /api/v1/chat/stream` SSE 流式对话接口。
- 流式响应完成后自动将用户提问与 AI 完整回答持久化至 SQLite，提供历史会话列表查询与级联删除接口，并完成全栈 `TestClient` 端到端验收。

**验收标准：**
- SQLite 数据库落盘后，即使重新创建新的 `create_app(db_url)` 实例，仍能完整读出此前的会话与消息历史。
- `POST /api/v1/chat/stream` 返回 `Content-Type: text/event-stream`，产出多帧合法 JSON `data:` 事件并以 `data: [DONE]` 收尾。
- 带 `Origin` 请求头的跨域请求能正确获得 `access-control-allow-origin` 响应头，且 `GET /` 返回可交互的星辰智脑 HTML 页面。

---

## 📚 课堂核心技术笔记

### 1. 1. SQLAlchemy 2.x 现代声明式 ORM 写法

SQLAlchemy 2.x 全面拥抱 Python 类型注解：
- 继承 `DeclarativeBase` 替代旧版 `declarative_base()`；
- 用 `id: Mapped[int] = mapped_column(primary_key=True)` 替代旧版 `Column(Integer, primary_key=True)`，IDE 自动补全与 mypy 类型检查体验大幅提升；
- 查询统一使用 `session.scalars(select(ChatMessage).where(...))`，彻底淘汰旧版 `session.query(...)`。

> **⚠️ 老陈避坑**：SQLite 线程坑：FastAPI 同步路由在多线程池中执行，SQLite 默认禁止跨线程复用连接，创建引擎时务必传入 `connect_args={'check_same_thread': False}`！

### 2. 2. SSE 流式响应与生产 Nginx 反向代理避坑

FastAPI 返回 `StreamingResponse(gen(), media_type='text/event-stream', headers={'Cache-Control': 'no-cache', 'X-Accel-Buffering': 'no'})`。
其中 `X-Accel-Buffering: no` 极其关键——线上环境前面通常挡着一层 Nginx，默认 Nginx 会把后端响应先攒够 4KB 再一起发给浏览器，导致你的 SSE 流式打字机在本地好好的、一上线就卡成一整坨！加上 `X-Accel-Buffering: no` 就能强制 Nginx 立刻逐帧透传！

> **⚠️ 老陈避坑**：SSE 每条消息必须以 `data: ` 开头、以**两个换行符 `\n\n`** 结尾，少写一个 `\n` 浏览器就不会触发消息解析。

### 3. 3. CORS 跨域安全配置要点

当网页运行在 `http://localhost:5173` 而 FastAPI 运行在 `http://localhost:8000` 时，协议/域名/端口任一不同即触发浏览器的同源策略（Same-Origin Policy）。生产环境应在 `CORSMiddleware` 中明确列出 `allow_origins=['https://starmind.example.com']`，切忌在 `allow_credentials=True` 时把 `allow_origins` 设为 `['*']`（浏览器会直接报错拒绝）。

> **⚠️ 老陈避坑**：排查 CORS 问题看浏览器开发者工具 Network 面板里的 `OPTIONS` 预检请求状态码和 `Access-Control-Allow-Origin` 响应头，一查一个准。


## 💻 实操步骤 1：步骤一：实现 SQLAlchemy 2.x + SQLite 对话历史持久化层

- **目标文件**：`versions/day24/src/day24_database.py`
- **老陈代码旁白**：咱们先建持久化地基 `src/day24_database.py`！严格采用 SQLAlchemy 2.x 现代 `DeclarativeBase` + `Mapped` 语法，建好 `ChatSession` 和 `ChatMessage` 一对多表结构，并封装干净的 `ChatRepository` 仓库类。


In [ ]:
"""
StarMind Day 24 - 步骤一：SQLAlchemy 2.x + SQLite 企业对话历史持久化层
支持文件 SQLite 与内存 SQLite，提供完整的会话与消息 CRUD 仓库接口。
"""

from datetime import datetime, timezone
from typing import Dict, List

from sqlalchemy import DateTime, ForeignKey, Integer, String, Text, create_engine, select
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, relationship, sessionmaker
from sqlalchemy.pool import StaticPool


def _utc_now() -> datetime:
    return datetime.now(timezone.utc)


class Base(DeclarativeBase):
    """SQLAlchemy 2.x 声明式模型基类。"""


class ChatSession(Base):
    """会话主表：记录企业员工创建的对话线程。"""

    __tablename__ = "chat_sessions"

    id: Mapped[str] = mapped_column(String(64), primary_key=True)
    title: Mapped[str] = mapped_column(String(120), default="新企业会话")
    created_at: Mapped[datetime] = mapped_column(DateTime(timezone=True), default=_utc_now)

    messages: Mapped[List["ChatMessage"]] = relationship(
        back_populates="session",
        cascade="all, delete-orphan",
        order_by="ChatMessage.id",
    )


class ChatMessage(Base):
    """消息明细表：持久化保存每一轮 user / assistant 文本。"""

    __tablename__ = "chat_messages"

    id: Mapped[int] = mapped_column(Integer, primary_key=True, autoincrement=True)
    session_id: Mapped[str] = mapped_column(
        String(64), ForeignKey("chat_sessions.id", ondelete="CASCADE"), index=True
    )
    role: Mapped[str] = mapped_column(String(16))
    content: Mapped[str] = mapped_column(Text)
    created_at: Mapped[datetime] = mapped_column(DateTime(timezone=True), default=_utc_now)

    session: Mapped[ChatSession] = relationship(back_populates="messages")


class ChatRepository:
    """面向 FastAPI 的对话历史持久化仓库。"""

    def __init__(self, database_url: str = "sqlite:///:memory:") -> None:
        connect_args = {"check_same_thread": False} if database_url.startswith("sqlite") else {}
        engine_kwargs = {"connect_args": connect_args}
        if database_url == "sqlite:///:memory:":
            engine_kwargs["poolclass"] = StaticPool

        self.engine = create_engine(database_url, **engine_kwargs)
        Base.metadata.create_all(self.engine)
        self.SessionLocal = sessionmaker(bind=self.engine, expire_on_commit=False)

    def ensure_session(self, session_id: str, title: str | None = None) -> ChatSession:
        """若会话不存在则自动创建，存在则直接返回。"""
        with self.SessionLocal() as db:
            sess = db.get(ChatSession, session_id)
            if sess is None:
                sess = ChatSession(id=session_id, title=(title or "星辰智脑会话")[:120])
                db.add(sess)
                db.commit()
                db.refresh(sess)
            return sess

    def add_message(self, session_id: str, role: str, content: str) -> ChatMessage:
        """追加一条对话消息到指定会话并提交落盘。"""
        with self.SessionLocal() as db:
            if db.get(ChatSession, session_id) is None:
                db.add(ChatSession(id=session_id, title=content[:30] or "星辰智脑会话"))
            msg = ChatMessage(session_id=session_id, role=role, content=content)
            db.add(msg)
            db.commit()
            db.refresh(msg)
            return msg

    def list_sessions(self) -> List[Dict[str, object]]:
        """列出全部历史会话摘要及消息条数。"""
        with self.SessionLocal() as db:
            sessions = db.scalars(select(ChatSession).order_by(ChatSession.created_at.desc())).all()
            return [
                {
                    "session_id": s.id,
                    "title": s.title,
                    "message_count": len(s.messages),
                }
                for s in sessions
            ]

    def get_messages(self, session_id: str) -> List[Dict[str, str]] | None:
        """查询指定会话的全部历史消息；若会话不存在返回 None。"""
        with self.SessionLocal() as db:
            sess = db.get(ChatSession, session_id)
            if sess is None:
                return None
            return [{"role": m.role, "content": m.content} for m in sess.messages]

    def delete_session(self, session_id: str) -> bool:
        """删除会话并级联清理其名下全部消息。"""
        with self.SessionLocal() as db:
            sess = db.get(ChatSession, session_id)
            if sess is None:
                return False
            db.delete(sess)
            db.commit()
            return True


if __name__ == "__main__":
    repo = ChatRepository("sqlite:///:memory:")
    repo.add_message("sess_01", "user", "你好，杭州出差住宿报销上限多少？")
    repo.add_message("sess_01", "assistant", "一线及新一线城市住宿上限 600 元/晚。")
    print("会话列表:", repo.list_sessions())
    print("消息历史:", repo.get_messages("sess_01"))


### 🔍 步骤 1 代码深度解析

1. 对 `sqlite:///:memory:` 自动配置 `StaticPool` 和 `check_same_thread=False`，保证内存数据库在多线程 `TestClient` 下共享同一张表而不会消失。
2. `ChatRepository.add_message` 在首次收到新 `session_id` 时自动创建会话并用首句提问前 30 字作为会话标题，完美复刻 ChatGPT 侧边栏体验。
3. `cascade='all, delete-orphan'` 确保删除会话时连带清理所有消息记录。

**预期运行输出参考：**
```text
会话列表: [{'session_id': 'sess_01', 'title': '你好，杭州出差住宿报销上限多少？', 'message_count': 2}]
消息历史: [{'role': 'user', 'content': '你好，杭州出差住宿报销上限多少？'}, {'role': 'assistant', 'content': '一线及新一线城市住宿上限 600 元/晚。'}]
```


## 💻 实操步骤 2：步骤二：实现 SSE 流式接口、CORS 跨域与网页版 ChatGPT 克隆全栈后端

- **目标文件**：`versions/day24/src/day24_chatgpt_backend.py`
- **老陈代码旁白**：压轴大戏 `src/day24_chatgpt_backend.py` 来啦！它把 `CORSMiddleware`、托管前端首页 `GET /`、`POST /api/v1/chat/stream`（SSE 流式打字机 + 自动落盘 SQLite）以及历史会话管理全部集成到一起，直接就是一套可上线内测的企业级 ChatGPT 克隆！


In [ ]:
"""
StarMind Day 24 - 步骤二：网页版星辰 ChatGPT 克隆全栈后端
集成 CORS 跨域中间件、静态前端首页托管、SSE 流式对话推送与 SQLite 持久化。
"""

import json
from typing import Dict, Generator, List

from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import HTMLResponse, StreamingResponse
from pydantic import BaseModel, ConfigDict, Field

from src.day24_database import ChatRepository


EMBEDDED_CHAT_HTML = """<!DOCTYPE html>
<html lang="zh-CN">
<head>
  <meta charset="UTF-8" />
  <title>星辰智脑 StarMind · 网页版企业 ChatGPT</title>
</head>
<body>
  <main id="starmind-app">
    <h1>星辰智脑 StarMind 企业工作台</h1>
    <div id="chat-log"></div>
    <form id="chat-form">
      <input id="chat-input" type="text" placeholder="请输入问题..." />
      <button id="send-btn" type="submit">发送</button>
    </form>
  </main>
  <script>
    const form = document.getElementById("chat-form");
    const input = document.getElementById("chat-input");
    const log = document.getElementById("chat-log");
    form.addEventListener("submit", async (e) => {
      e.preventDefault();
      const text = input.value.trim();
      if (!text) return;
      input.value = "";
      const resp = await fetch("/api/v1/chat/stream", {
        method: "POST",
        headers: {"Content-Type": "application/json"},
        body: JSON.stringify({session_id: "web_default", message: text})
      });
      const reader = resp.body.getReader();
      const decoder = new TextDecoder("utf-8");
      const p = document.createElement("p");
      log.appendChild(p);
      while (true) {
        const {value, done} = await reader.read();
        if (done) break;
        const lines = decoder.decode(value).split("\\n");
        for (const line of lines) {
          if (line.startsWith("data: ") && !line.includes("[DONE]")) {
            const obj = JSON.parse(line.slice(6));
            if (obj.delta) p.textContent += obj.delta;
          }
        }
      }
    });
  </script>
</body>
</html>
"""


class StreamChatRequest(BaseModel):
    model_config = ConfigDict(str_strip_whitespace=True)

    session_id: str = Field(..., min_length=2, max_length=64)
    message: str = Field(..., min_length=1, max_length=4000)
    chunk_size: int = Field(default=6, ge=1, le=64)


def _synthesize_stream_reply(message: str, history: List[Dict[str, str]]) -> str:
    """结合 SQLite 持久化历史生成上下文连贯的回复文本。"""
    if "差旅" in message or "住宿" in message:
        return "根据星辰数科差旅报销规范：杭州/北京/上海/深圳住宿上限为 600 元/晚，请保留电子专票。"
    if "历史" in message or "上一句" in message:
        user_msgs = [m["content"] for m in history if m["role"] == "user"]
        if len(user_msgs) >= 2:
            return f"我从数据库历史中查到，您上一条问题是：「{user_msgs[-2]}」。"
    return f"星辰智脑已收到您的消息「{message}」，本次对话已实时流式推送并持久化保存至 SQLite。"


def create_chatgpt_app(
    database_url: str = "sqlite:///:memory:",
    allowed_origins: List[str] | None = None,
) -> FastAPI:
    """构建完整可联调的网页版星辰 ChatGPT 克隆后端应用。"""
    app = FastAPI(title="星辰智脑 ChatGPT 克隆全栈服务", version="1.0.0")
    repo = ChatRepository(database_url=database_url)

    # 1. 配置企业级 CORS 跨域中间件
    origins = allowed_origins or ["http://localhost:3000", "http://localhost:5173", "https://starmind.local"]
    app.add_middleware(
        CORSMiddleware,
        allow_origins=origins,
        allow_credentials=True,
        allow_methods=["*"],
        allow_headers=["*"],
    )

    # 2. 托管静态前端聊天首页
    @app.get("/", response_class=HTMLResponse, tags=["Web"])
    def serve_index() -> HTMLResponse:
        return HTMLResponse(content=EMBEDDED_CHAT_HTML)

    # 3. SSE 流式对话接口 + 自动持久化落盘
    @app.post("/api/v1/chat/stream", tags=["Chat"])
    def stream_chat(req: StreamChatRequest) -> StreamingResponse:
        # 先落盘用户提问，确保即使后续中断也能追溯请求
        repo.add_message(req.session_id, "user", req.message)
        history = repo.get_messages(req.session_id) or []
        full_reply = _synthesize_stream_reply(req.message, history)

        def sse_generator() -> Generator[str, None, None]:
            collected: List[str] = []
            for idx in range(0, len(full_reply), req.chunk_size):
                piece = full_reply[idx : idx + req.chunk_size]
                collected.append(piece)
                payload = json.dumps(
                    {"session_id": req.session_id, "delta": piece},
                    ensure_ascii=False,
                )
                yield f"data: {payload}\n\n"

            # 流式推送完毕，将完整助手回复持久化写入 SQLite
            repo.add_message(req.session_id, "assistant", "".join(collected))
            yield "data: [DONE]\n\n"

        return StreamingResponse(
            sse_generator(),
            media_type="text/event-stream",
            headers={
                "Cache-Control": "no-cache",
                "X-Accel-Buffering": "no",
            },
        )

    # 4. 历史会话列表、详情与级联删除接口
    @app.get("/api/v1/sessions", tags=["Sessions"])
    def list_sessions() -> Dict[str, object]:
        return {"sessions": repo.list_sessions()}

    @app.get("/api/v1/sessions/{session_id}", tags=["Sessions"])
    def get_session_history(session_id: str) -> Dict[str, object]:
        msgs = repo.get_messages(session_id)
        if msgs is None:
            raise HTTPException(status_code=404, detail=f"会话 '{session_id}' 不存在")
        return {"session_id": session_id, "messages": msgs}

    @app.delete("/api/v1/sessions/{session_id}", tags=["Sessions"])
    def delete_session(session_id: str) -> Dict[str, str]:
        if not repo.delete_session(session_id):
            raise HTTPException(status_code=404, detail=f"会话 '{session_id}' 不存在")
        return {"status": "deleted", "session_id": session_id}

    return app


app = create_chatgpt_app()


if __name__ == "__main__":
    from fastapi.testclient import TestClient

    client = TestClient(create_chatgpt_app())
    with client.stream("POST", "/api/v1/chat/stream", json={"session_id": "web_01", "message": "杭州差旅住宿报销标准？"}) as r:
        print("Content-Type:", r.headers["content-type"])
        for line in r.iter_lines():
            if line:
                print("SSE帧 ->", line)
    print("持久化查询:", client.get("/api/v1/sessions/web_01").json())


### 🔍 步骤 2 代码深度解析

1. `StreamingResponse` 配合 `media_type='text/event-stream'` 与 `X-Accel-Buffering: no`，完美满足生产环境流式透传规范。
2. 生成器内部在推送全部增量帧后，立刻调用 `repo.add_message(..., 'assistant', ...)` 落盘，随后发送标准 `data: [DONE]\n\n` 终止标志。
3. 通过参数化 `database_url`，单测时可以传入真实的 `sqlite:///{tmp_path}/starmind.db` 文件验证**跨进程重启持久化**！

**预期运行输出参考：**
```text
Content-Type: text/event-stream; charset=utf-8
SSE帧 -> data: {"session_id": "web_01", "delta": "根据星辰数科"}
...
SSE帧 -> data: [DONE]
持久化查询: {"session_id": "web_01", "messages": [{"role": "user", ...}, {"role": "assistant", ...}]}
```


## 🧪 当日自动化单元测试

在终端运行 `pytest versions/day24/tests/test_day24.py -v` 即可验收全部测试断言。下面展示完整测试脚本源码，供对照学习测试写法：

```python
"""Day 24 自动化单元测试：端到端验证 SSE 流式推送、CORS 跨域、SQLite 跨重启持久化与前端页面托管。"""

import json
from pathlib import Path
from fastapi.testclient import TestClient
from src.day24_chatgpt_backend import create_chatgpt_app


def test_frontend_html_and_cors_headers():
    client = TestClient(create_chatgpt_app())

    # 1. 验证前端聊天首页托管
    r_home = client.get("/")
    assert r_home.status_code == 200
    assert "text/html" in r_home.headers["content-type"]
    assert "/api/v1/chat/stream" in r_home.text

    # 2. 验证 CORS 跨域响应头
    r_cors = client.options(
        "/api/v1/chat/stream",
        headers={
            "Origin": "http://localhost:5173",
            "Access-Control-Request-Method": "POST",
        },
    )
    assert r_cors.status_code == 200
    assert r_cors.headers.get("access-control-allow-origin") == "http://localhost:5173"


def test_sse_stream_and_sqlite_persistence_across_restarts(tmp_path: Path):
    db_file = tmp_path / "starmind_day24.db"
    db_url = f"sqlite:///{db_file}"

    # 第一次启动应用并发送 SSE 流式对话
    client1 = TestClient(create_chatgpt_app(database_url=db_url))
    chunks = []
    saw_done = False
    with client1.stream(
        "POST",
        "/api/v1/chat/stream",
        json={"session_id": "persist_01", "message": "杭州出差住宿报销标准是多少？", "chunk_size": 8},
    ) as resp:
        assert resp.status_code == 200
        assert "text/event-stream" in resp.headers["content-type"]
        for line in resp.iter_lines():
            if line.startswith("data: "):
                raw = line[6:].strip()
                if raw == "[DONE]":
                    saw_done = True
                else:
                    chunks.append(json.loads(raw)["delta"])

    assembled = "".join(chunks)
    assert saw_done is True
    assert len(chunks) >= 3
    assert "600 元/晚" in assembled

    # 模拟服务重启：使用同一个 SQLite 文件重建全新 FastAPI 实例
    client2 = TestClient(create_chatgpt_app(database_url=db_url))
    r_hist = client2.get("/api/v1/sessions/persist_01")
    assert r_hist.status_code == 200
    msgs = r_hist.json()["messages"]
    assert len(msgs) == 2
    assert msgs[0]["role"] == "user"
    assert msgs[1]["role"] == "assistant"
    assert msgs[1]["content"] == assembled

    # 验证多轮跨重启记忆
    with client2.stream(
        "POST",
        "/api/v1/chat/stream",
        json={"session_id": "persist_01", "message": "我上一句问了什么？"},
    ) as resp2:
        text2 = "".join(
            json.loads(line[6:])["delta"]
            for line in resp2.iter_lines()
            if line.startswith("data: ") and line[6:].strip() != "[DONE]"
        )
    assert "杭州出差住宿报销标准" in text2

    # 验证级联删除
    assert client2.delete("/api/v1/sessions/persist_01").status_code == 200
    assert client2.get("/api/v1/sessions/persist_01").status_code == 404

```


## 📝 课后梯度实战作业与标准答案

建议先在下方新建空白单元格独立完成编码，运行通过后再核对参考答案。


### 作业 1（基础巩固）：为 ChatRepository 增加会话标题重命名方法

**题目要求：**
编写辅助函数 `rename_session(repo, session_id, new_title)`，更新 SQLite 中对应 `ChatSession.title` 字段，若会话不存在返回 `False`，更新成功返回 `True`。

**思路提示：** 使用 `with repo.SessionLocal() as db:` 查出 `ChatSession`，修改 `.title` 后 `db.commit()`。


In [ ]:
from src.day24_database import ChatRepository, ChatSession


def rename_session(repo: ChatRepository, session_id: str, new_title: str) -> bool:
    """更新指定会话的展示标题并持久化到 SQLite。"""
    with repo.SessionLocal() as db:
        sess = db.get(ChatSession, session_id)
        if sess is None:
            return False
        sess.title = new_title.strip()[:120]
        db.commit()
        return True


if __name__ == "__main__":
    repo = ChatRepository("sqlite:///:memory:")
    repo.add_message("s_rename", "user", "初始提问")
    print("重命名结果:", rename_session(repo, "s_rename", "2026 差旅报销咨询"))
    print("最新列表:", repo.list_sessions())


**答案解析与评分标准：**

通过 SQLAlchemy 会话上下文管理器执行属性修改与 `commit()`，即可自动生成参数化 `UPDATE` 语句防止 SQL 注入。


### 作业 2（企业实战）：按关键词全文检索历史消息（SQLite LIKE 查询）

**题目要求：**
编写 `search_messages(repo, keyword)`，使用 SQLAlchemy 2.x `select(ChatMessage).where(ChatMessage.content.contains(keyword))` 检索包含指定关键词的所有历史消息，返回 `session_id`、`role`、`content` 字典列表。

**思路提示：** 调用 `db.scalars(select(ChatMessage).where(ChatMessage.content.contains(keyword))).all()`。


In [ ]:
from typing import Dict, List
from sqlalchemy import select
from src.day24_database import ChatMessage, ChatRepository


def search_messages(repo: ChatRepository, keyword: str) -> List[Dict[str, str]]:
    """使用 SQLAlchemy 2.x 在全量聊天记录中按关键词模糊检索。"""
    with repo.SessionLocal() as db:
        stmt = select(ChatMessage).where(ChatMessage.content.contains(keyword)).order_by(ChatMessage.id.desc())
        rows = db.scalars(stmt).all()
        return [{"session_id": r.session_id, "role": r.role, "content": r.content} for r in rows]


if __name__ == "__main__":
    repo = ChatRepository("sqlite:///:memory:")
    repo.add_message("s1", "user", "请问 VPN 怎么开通？")
    repo.add_message("s2", "user", "杭州差旅住宿报销标准？")
    print("搜索 VPN:", search_messages(repo, "VPN"))


**答案解析与评分标准：**

SQLAlchemy 的 `.contains()` 会自动转义通配符并使用绑定参数，比手拼 `f"LIKE '%{keyword}%'"` 安全得多。


### 作业 3（架构进阶）：SSE 心跳保活（Heartbeat）流式生成器包装器

**题目要求：**
当大模型深度思考（如 DeepSeek-R1）耗时超过 30 秒才吐第一个字时，负载均衡器（SLB/Nginx）可能会主动断开空闲连接。请编写生成器 `with_sse_heartbeat(chunks)`，在首个真实数据块之前先发送一条 SSE 注释心跳帧 `: keep-alive\n\n`，再逐帧透传业务数据。

**思路提示：** SSE 协议规定以冒号 `:` 开头的行属于注释帧，前端解析器会自动忽略，但能有效刷新网关空闲超时计时器。


In [ ]:
from typing import Generator, Iterable


def with_sse_heartbeat(data_frames: Iterable[str]) -> Generator[str, None, None]:
    """在 SSE 流开头注入标准注释心跳帧，防止长推理期间代理网关超时断连。"""
    yield ": keep-alive\n\n"
    for frame in data_frames:
        yield frame


if __name__ == "__main__":
    sample = ['data: {"delta": "你好"}\n\n', "data: [DONE]\n\n"]
    print(list(with_sse_heartbeat(sample)))


**答案解析与评分标准：**

这是生产级大模型流式网关解决「长思考导致 504 Gateway Timeout」最经典、零副作用的工程技巧。


---

## 🎯 老陈收尾复盘

掌声送给自己！从 Day 15 的第一行 Token 估算器，到今天 Day 24 跑通带 SQLite 持久化、SSE 流式打字机、CORS 跨域和 Web 聊天界面的「网页版星辰 ChatGPT 克隆」，第二阶段 10 天硬核实战圆满收官！大家已经彻底具备了独立开发全栈 AI 应用的能力。今晚好好庆祝一下，明天 Day 25 咱们正式开启第三阶段——LangChain 与企业级 RAG 知识库开发！
